# Understanding Passenger Satisfaction Through Advanced Text Mining

**Group 5:** Bansi Khachar, Lucynda Young, Matthew Prado, Saurabh Parate  
**Dataset:** Skytrax Airline Reviews  
**Source:** https://www.kaggle.com/datasets/efehandanisman/skytrax-airline-reviews

## Abstract

This project applies exploratory data analysis, topic modeling, sentiment analysis, and supervised classification to Skytrax airline reviews. The study aims to identify the service themes most frequently discussed by passengers, compare multiple NLP approaches, evaluate sentiment trends, and predict whether a passenger recommends an airline from the review text. The notebook is organized as a reproducible workflow from data loading and cleaning through model comparison and business interpretation.

## 0. Environment Setup

Install packages only when they are not already available. Restart the kernel after installation if necessary.

In [ ]:
# Uncomment this cell only if packages are missing.
# %pip install pandas numpy matplotlib scikit-learn nltk spacy textblob vaderSentiment
# %pip install gensim wordcloud seaborn
# %pip install bertopic sentence-transformers transformers torch
# %pip install imbalanced-learn xgboost
# %pip install kaleido

# Download the English spaCy model from a terminal or notebook cell if needed:
# !python -m spacy download en_core_web_sm

In [ ]:
import os
import re
import html
import string
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 150)

RANDOM_STATE = 42
DATA_PATH = Path("data")  # Place the downloaded Kaggle CSV file(s) in this folder.
OUTPUT_PATH = Path("outputs")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# 1. Data Loading

This section locates the CSV file, loads it into a DataFrame, inspects the available columns, and standardizes column names. The code is designed to tolerate minor differences in the downloaded Kaggle filename.

In [ ]:
csv_files = list(DATA_PATH.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(
        "No CSV file was found. Download the Kaggle dataset and place the CSV file "
        "inside a folder named 'data' next to this notebook."
    )

print("CSV files found:")
for file in csv_files:
    print("-", file)

# Use the largest CSV file as the primary dataset.
dataset_file = max(csv_files, key=lambda p: p.stat().st_size)
print(f"\nLoading: {dataset_file}")

df_raw = pd.read_csv(dataset_file, low_memory=False)
print(f"Original shape: {df_raw.shape}")
display(df_raw.head())

In [ ]:
def standardize_column_name(column_name):
    column_name = str(column_name).strip().lower()
    column_name = re.sub(r"[^a-z0-9]+", "_", column_name)
    return column_name.strip("_")

df = df_raw.copy()
df.columns = [standardize_column_name(col) for col in df.columns]

print("Standardized columns:")
print(df.columns.tolist())
df.info()

## 1.1 Identify Core Variables

The dataset may use slightly different field names depending on the downloaded version. This helper function identifies likely columns for review text, recommendation status, ratings, airline, cabin class, traveler type, and dates.

In [ ]:
def find_first_matching_column(dataframe, candidates):
    columns = set(dataframe.columns)
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None

TEXT_COL = find_first_matching_column(
    df,
    ["customer_review", "review", "review_text", "content", "text", "comments"]
)

TITLE_COL = find_first_matching_column(
    df,
    ["review_title", "title", "headline"]
)

RECOMMEND_COL = find_first_matching_column(
    df,
    ["recommended", "recommendation", "recommend", "is_recommended"]
)

RATING_COL = find_first_matching_column(
    df,
    ["overall_rating", "rating", "review_rating", "score"]
)

AIRLINE_COL = find_first_matching_column(
    df,
    ["airline_name", "airline", "company", "airline_slug"]
)

TRAVELER_COL = find_first_matching_column(
    df,
    ["traveller_type", "traveler_type", "type_of_traveller", "type_of_traveler"]
)

CABIN_COL = find_first_matching_column(
    df,
    ["cabin_flown", "cabin_class", "seat_type", "class"]
)

DATE_COL = find_first_matching_column(
    df,
    ["date_flown", "review_date", "date_published", "date", "published_date"]
)

COLUMN_MAP = {
    "text": TEXT_COL,
    "title": TITLE_COL,
    "recommendation": RECOMMEND_COL,
    "rating": RATING_COL,
    "airline": AIRLINE_COL,
    "traveler_type": TRAVELER_COL,
    "cabin_class": CABIN_COL,
    "date": DATE_COL,
}

print("Detected core columns:")
for key, value in COLUMN_MAP.items():
    print(f"{key:15}: {value}")

if TEXT_COL is None:
    raise ValueError(
        "A review-text column was not detected. Inspect df.columns and manually set TEXT_COL."
    )

# 2. Data Cleaning and Preprocessing

Two text versions are retained:

- `review_text_original`: minimally cleaned text used for transformer sentiment and qualitative interpretation.
- `review_text_clean`: normalized text used for frequency analysis, traditional topic modeling, and classification.

The cleaning workflow removes missing text, exact duplicates, HTML artifacts, URLs, punctuation, isolated numbers, stopwords, and very short tokens while preserving airline-related terminology.

In [ ]:
cleaning_log = []

def log_step(step_name, before_count, after_count):
    cleaning_log.append({
        "step": step_name,
        "records_before": before_count,
        "records_after": after_count,
        "records_removed": before_count - after_count,
    })

starting_count = len(df)

# Preserve only records with usable text.
before = len(df)
df = df[df[TEXT_COL].notna()].copy()
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df = df[df[TEXT_COL].ne("")].copy()
log_step("Remove missing or empty review text", before, len(df))

# Remove exact duplicate text records.
before = len(df)
df = df.drop_duplicates(subset=[TEXT_COL]).copy()
log_step("Remove duplicate review text", before, len(df))

# Reset index after row filtering.
df = df.reset_index(drop=True)

print(f"Original records: {starting_count:,}")
print(f"Records after initial filtering: {len(df):,}")
display(pd.DataFrame(cleaning_log))

In [ ]:
# Normalize key structured variables.

if RATING_COL:
    df[RATING_COL] = pd.to_numeric(df[RATING_COL], errors="coerce")

if DATE_COL:
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

def normalize_recommendation(value):
    if pd.isna(value):
        return np.nan

    normalized = str(value).strip().lower()

    positive_values = {"yes", "y", "true", "1", "recommended", "recommend"}
    negative_values = {"no", "n", "false", "0", "not recommended", "not_recommended"}

    if normalized in positive_values:
        return 1
    if normalized in negative_values:
        return 0

    # Handle numeric-like values.
    try:
        numeric = float(normalized)
        if numeric == 1:
            return 1
        if numeric == 0:
            return 0
    except ValueError:
        pass

    return np.nan

if RECOMMEND_COL:
    df["recommended_binary"] = df[RECOMMEND_COL].apply(normalize_recommendation)
    print(df[[RECOMMEND_COL, "recommended_binary"]].head())
else:
    print("Recommendation column not detected. Classification will require manual column mapping.")

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

for resource in ["stopwords", "wordnet", "omw-1.4", "punkt", "averaged_perceptron_tagger"]:
    try:
        nltk.download(resource, quiet=True)
    except Exception:
        pass

STOP_WORDS = set(stopwords.words("english"))

# Retain negations because they affect sentiment.
STOP_WORDS -= {"no", "nor", "not", "never"}

# Add generic review/platform words that are not analytically useful.
CUSTOM_STOP_WORDS = {
    "airline", "airlines", "flight", "flights", "fly", "flying",
    "passenger", "passengers", "review", "reviews", "skytrax",
    "would", "could", "also", "one", "get", "got"
}
STOP_WORDS |= CUSTOM_STOP_WORDS

lemmatizer = WordNetLemmatizer()

def minimally_clean_text(text):
    text = html.unescape(str(text))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def preprocess_text(text):
    text = minimally_clean_text(text).lower()

    # Preserve common hyphenated aviation expressions by replacing hyphens with spaces.
    text = text.replace("-", " ")

    # Remove punctuation and isolated numbers.
    text = re.sub(r"[^a-z\s']", " ", text)
    text = re.sub(r"\b\d+\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = [
        token for token in text.split()
        if len(token) > 2 and token not in STOP_WORDS
    ]

    lemmas = [lemmatizer.lemmatize(token) for token in tokens]
    return " ".join(lemmas)

df["review_text_original"] = df[TEXT_COL].apply(minimally_clean_text)
df["review_text_clean"] = df["review_text_original"].apply(preprocess_text)
df["word_count_original"] = df["review_text_original"].str.split().str.len()
df["word_count_clean"] = df["review_text_clean"].str.split().str.len()
df["character_count"] = df["review_text_original"].str.len()

before = len(df)
df = df[df["word_count_clean"] >= 3].copy()
log_step("Remove reviews with fewer than 3 cleaned tokens", before, len(df))

df = df.reset_index(drop=True)

print(f"Final cleaned records: {len(df):,}")
display(df[[TEXT_COL, "review_text_clean", "word_count_original"]].head())

In [ ]:
cleaning_summary = pd.DataFrame(cleaning_log)
display(cleaning_summary)

cleaning_summary.to_csv(OUTPUT_PATH / "cleaning_summary.csv", index=False)
df.to_csv(OUTPUT_PATH / "skytrax_airline_reviews_cleaned.csv", index=False)

print("Cleaned dataset saved to:", OUTPUT_PATH / "skytrax_airline_reviews_cleaned.csv")

# 3. Exploratory Data Analysis

The EDA examines data quality, review-length distributions, structured rating variables, recommendation behavior, frequent words, part-of-speech patterns, and named entities.

In [ ]:
print("Cleaned dataset shape:", df.shape)
display(df.head())

missing_summary = (
    df.isna()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
      .rename("missing_percent")
      .to_frame()
)

display(missing_summary.head(20))

## 3.1 Descriptive Statistics

In [ ]:
numeric_summary_columns = [
    col for col in [
        RATING_COL,
        "word_count_original",
        "word_count_clean",
        "character_count"
    ]
    if col is not None and col in df.columns
]

display(df[numeric_summary_columns].describe().T)

## 3.2 Review Length Distribution

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df["word_count_original"].dropna(), bins=50)
plt.title("Distribution of Airline Review Lengths")
plt.xlabel("Words per Review")
plt.ylabel("Number of Reviews")
plt.tight_layout()
plt.show()

print("Median review length:", int(df["word_count_original"].median()))
print("Mean review length:", round(df["word_count_original"].mean(), 2))

## 3.3 Overall Rating Distribution

In [ ]:
if RATING_COL:
    rating_counts = df[RATING_COL].value_counts(dropna=False).sort_index()

    plt.figure(figsize=(9, 5))
    rating_counts.plot(kind="bar")
    plt.title("Distribution of Overall Airline Ratings")
    plt.xlabel("Overall Rating")
    plt.ylabel("Number of Reviews")
    plt.tight_layout()
    plt.show()

    display(rating_counts.rename("review_count").to_frame())
else:
    print("No overall-rating column was detected.")

## 3.4 Recommendation Distribution

In [ ]:
if "recommended_binary" in df.columns:
    recommendation_counts = (
        df["recommended_binary"]
        .map({1: "Recommended", 0: "Not Recommended"})
        .value_counts(dropna=False)
    )

    plt.figure(figsize=(7, 5))
    recommendation_counts.plot(kind="bar")
    plt.title("Recommendation Status Distribution")
    plt.xlabel("Recommendation Status")
    plt.ylabel("Number of Reviews")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

    display(recommendation_counts.rename("review_count").to_frame())
else:
    print("Recommendation status is unavailable.")

## 3.5 Traveler Type and Cabin Class

In [ ]:
categorical_columns = {
    "Traveler Type": TRAVELER_COL,
    "Cabin Class": CABIN_COL,
    "Airline": AIRLINE_COL,
}

for label, column in categorical_columns.items():
    if column:
        counts = df[column].fillna("Missing").value_counts().head(15)

        plt.figure(figsize=(10, 5))
        counts.sort_values().plot(kind="barh")
        plt.title(f"Top {label} Categories")
        plt.xlabel("Number of Reviews")
        plt.ylabel(label)
        plt.tight_layout()
        plt.show()

## 3.6 Most Frequent Words

In [ ]:
all_tokens = " ".join(df["review_text_clean"].dropna()).split()
word_counts = Counter(all_tokens)
top_words = pd.DataFrame(word_counts.most_common(30), columns=["word", "count"])

display(top_words)

plt.figure(figsize=(10, 7))
plt.barh(top_words["word"][::-1], top_words["count"][::-1])
plt.title("Thirty Most Frequent Terms in Cleaned Reviews")
plt.xlabel("Frequency")
plt.ylabel("Term")
plt.tight_layout()
plt.show()

## 3.7 Word Cloud

In [ ]:
from wordcloud import WordCloud

wordcloud = WordCloud(
    width=1400,
    height=700,
    background_color="white",
    collocations=False,
    random_state=RANDOM_STATE
).generate(" ".join(all_tokens))

plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud of Skytrax Airline Reviews")
plt.tight_layout()
plt.show()

## 3.8 Part-of-Speech Distribution

To control runtime, POS tagging is performed on a reproducible sample of reviews.

In [ ]:
import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError as error:
    raise OSError(
        "spaCy English model is missing. Run: python -m spacy download en_core_web_sm"
    ) from error

eda_sample = df["review_text_original"].dropna().sample(
    n=min(1500, df["review_text_original"].notna().sum()),
    random_state=RANDOM_STATE
)

pos_counter = Counter()

for document in nlp.pipe(eda_sample.tolist(), batch_size=64):
    for token in document:
        if not token.is_space and not token.is_punct:
            pos_counter[token.pos_] += 1

pos_df = pd.DataFrame(pos_counter.most_common(), columns=["part_of_speech", "count"])
display(pos_df)

plt.figure(figsize=(9, 5))
plt.bar(pos_df["part_of_speech"], pos_df["count"])
plt.title("Part-of-Speech Distribution in Review Sample")
plt.xlabel("Part of Speech")
plt.ylabel("Token Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3.9 Named Entity Recognition

In [ ]:
entity_counter = Counter()

for document in nlp.pipe(eda_sample.tolist(), batch_size=64):
    for entity in document.ents:
        cleaned_entity = entity.text.strip()
        if len(cleaned_entity) > 1:
            entity_counter[(cleaned_entity, entity.label_)] += 1

entity_df = pd.DataFrame(
    [
        {"entity": entity, "entity_type": entity_type, "count": count}
        for (entity, entity_type), count in entity_counter.most_common(30)
    ]
)

display(entity_df)

# 4. Feature Engineering

Traditional models use TF-IDF text features. The classification target is recommendation status, where 1 represents recommended and 0 represents not recommended.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

classification_df = df.dropna(subset=["review_text_clean"]).copy()

if "recommended_binary" in classification_df.columns:
    classification_df = classification_df.dropna(subset=["recommended_binary"]).copy()
    classification_df["recommended_binary"] = classification_df["recommended_binary"].astype(int)

    X_train_text, X_test_text, y_train, y_test = train_test_split(
        classification_df["review_text_clean"],
        classification_df["recommended_binary"],
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=classification_df["recommended_binary"],
    )

    tfidf_classifier = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.95,
        max_features=30000,
        sublinear_tf=True,
    )

    X_train_tfidf = tfidf_classifier.fit_transform(X_train_text)
    X_test_tfidf = tfidf_classifier.transform(X_test_text)

    print("Training matrix shape:", X_train_tfidf.shape)
    print("Testing matrix shape:", X_test_tfidf.shape)
    print("\nTraining target distribution:")
    print(y_train.value_counts(normalize=True))
else:
    print("Classification target unavailable. Map the recommendation column before continuing.")

# 5. Topic Modeling

LDA, NMF, and BERTopic are tested and compared. Traditional coherence scores are calculated for LDA and NMF. BERTopic is evaluated through topic diversity, topic size, and qualitative interpretability.

## 5.1 Shared Topic Modeling Data

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

topic_texts = df["review_text_clean"].dropna()
topic_texts = topic_texts[topic_texts.str.split().str.len() >= 5]

# Use a reproducible sample if the dataset is extremely large.
MAX_TOPIC_DOCUMENTS = 25000
if len(topic_texts) > MAX_TOPIC_DOCUMENTS:
    topic_texts = topic_texts.sample(MAX_TOPIC_DOCUMENTS, random_state=RANDOM_STATE)

topic_documents = topic_texts.tolist()
tokenized_topic_documents = [text.split() for text in topic_documents]

print("Documents used for topic modeling:", len(topic_documents))

## 5.2 LDA Topic Modeling

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

count_vectorizer = CountVectorizer(
    min_df=5,
    max_df=0.90,
    max_features=15000,
    ngram_range=(1, 2),
)

count_matrix = count_vectorizer.fit_transform(topic_documents)
count_feature_names = np.array(count_vectorizer.get_feature_names_out())

def extract_top_words(model, feature_names, number_of_words=12):
    topics = {}
    for topic_index, component in enumerate(model.components_):
        top_indices = component.argsort()[-number_of_words:][::-1]
        topics[topic_index] = feature_names[top_indices].tolist()
    return topics

lda_models = {}
lda_results = []

for number_of_topics in [5, 7, 10, 12]:
    lda_model = LatentDirichletAllocation(
        n_components=number_of_topics,
        learning_method="batch",
        max_iter=20,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    lda_model.fit(count_matrix)

    lda_models[number_of_topics] = lda_model
    lda_results.append({
        "number_of_topics": number_of_topics,
        "perplexity": lda_model.perplexity(count_matrix),
        "log_likelihood": lda_model.score(count_matrix),
    })

lda_comparison = pd.DataFrame(lda_results)
display(lda_comparison)

In [ ]:
# Select the topic count after reviewing perplexity and interpretability.
SELECTED_LDA_TOPICS = 7
selected_lda = lda_models[SELECTED_LDA_TOPICS]
lda_topics = extract_top_words(selected_lda, count_feature_names)

for topic_number, words in lda_topics.items():
    print(f"LDA Topic {topic_number + 1}: {', '.join(words)}")

## 5.3 NMF Topic Modeling

In [ ]:
from sklearn.decomposition import NMF

topic_tfidf_vectorizer = TfidfVectorizer(
    min_df=5,
    max_df=0.90,
    max_features=15000,
    ngram_range=(1, 2),
)

topic_tfidf_matrix = topic_tfidf_vectorizer.fit_transform(topic_documents)
topic_tfidf_feature_names = np.array(topic_tfidf_vectorizer.get_feature_names_out())

nmf_models = {}
nmf_results = []

for number_of_topics in [5, 7, 10, 12]:
    nmf_model = NMF(
        n_components=number_of_topics,
        init="nndsvda",
        random_state=RANDOM_STATE,
        max_iter=500,
    )
    document_topic_matrix = nmf_model.fit_transform(topic_tfidf_matrix)

    nmf_models[number_of_topics] = nmf_model
    nmf_results.append({
        "number_of_topics": number_of_topics,
        "reconstruction_error": nmf_model.reconstruction_err_,
    })

nmf_comparison = pd.DataFrame(nmf_results)
display(nmf_comparison)

In [ ]:
SELECTED_NMF_TOPICS = 7
selected_nmf = nmf_models[SELECTED_NMF_TOPICS]
nmf_topics = extract_top_words(selected_nmf, topic_tfidf_feature_names)

for topic_number, words in nmf_topics.items():
    print(f"NMF Topic {topic_number + 1}: {', '.join(words)}")

## 5.4 Topic Coherence for LDA and NMF

In [ ]:
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

gensim_dictionary = Dictionary(tokenized_topic_documents)

def calculate_coherence(topic_word_dictionary, tokenized_documents, dictionary):
    topic_word_lists = list(topic_word_dictionary.values())
    coherence_model = CoherenceModel(
        topics=topic_word_lists,
        texts=tokenized_documents,
        dictionary=dictionary,
        coherence="c_v",
    )
    return coherence_model.get_coherence()

traditional_topic_comparison = []

for number_of_topics, model in lda_models.items():
    topics = extract_top_words(model, count_feature_names)
    coherence = calculate_coherence(
        topics,
        tokenized_topic_documents,
        gensim_dictionary,
    )
    traditional_topic_comparison.append({
        "model": "LDA",
        "number_of_topics": number_of_topics,
        "coherence_cv": coherence,
    })

for number_of_topics, model in nmf_models.items():
    topics = extract_top_words(model, topic_tfidf_feature_names)
    coherence = calculate_coherence(
        topics,
        tokenized_topic_documents,
        gensim_dictionary,
    )
    traditional_topic_comparison.append({
        "model": "NMF",
        "number_of_topics": number_of_topics,
        "coherence_cv": coherence,
    })

traditional_topic_comparison = pd.DataFrame(traditional_topic_comparison)
display(traditional_topic_comparison.sort_values("coherence_cv", ascending=False))

## 5.5 BERTopic

BERTopic can require substantial memory and processing time. A capped sample is used by default.

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

MAX_BERTOPIC_DOCUMENTS = 10000
bertopic_texts = df["review_text_original"].dropna()

if len(bertopic_texts) > MAX_BERTOPIC_DOCUMENTS:
    bertopic_texts = bertopic_texts.sample(
        MAX_BERTOPIC_DOCUMENTS,
        random_state=RANDOM_STATE
    )

bertopic_documents = bertopic_texts.tolist()

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
bertopic_vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
)

bertopic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=bertopic_vectorizer,
    calculate_probabilities=False,
    verbose=True,
)

bertopic_topics, bertopic_probabilities = bertopic_model.fit_transform(
    bertopic_documents
)

bertopic_info = bertopic_model.get_topic_info()
display(bertopic_info.head(20))

In [ ]:
# Display the top terms for the largest non-outlier BERTopic topics.
for topic_id in bertopic_info.loc[bertopic_info["Topic"] != -1, "Topic"].head(10):
    print(f"BERTopic {topic_id}:")
    print(bertopic_model.get_topic(topic_id))
    print()

In [ ]:
# Optional BERTopic visualizations.
# bertopic_model.visualize_barchart(top_n_topics=10)
# bertopic_model.visualize_topics()
# bertopic_model.visualize_hierarchy()

## 5.6 Topic Model Comparison

Complete the interpretability scores after the team reviews sample documents and topic labels.

In [ ]:
topic_model_summary = pd.DataFrame({
    "model": ["LDA", "NMF", "BERTopic"],
    "quantitative_measure": [
        traditional_topic_comparison.query("model == 'LDA'")["coherence_cv"].max(),
        traditional_topic_comparison.query("model == 'NMF'")["coherence_cv"].max(),
        np.nan,
    ],
    "human_interpretability_score_1_to_5": [np.nan, np.nan, np.nan],
    "notes": [
        "Add interpretation notes.",
        "Add interpretation notes.",
        "Evaluate semantic quality, outlier rate, and topic distinctiveness.",
    ],
})

display(topic_model_summary)

# 6. Sentiment Analysis

TextBlob, VADER, and a transformer-based sentiment model are compared. Human-coded validation is included to support model selection.

## 6.1 TextBlob

In [ ]:
from textblob import TextBlob

def textblob_sentiment(text):
    polarity = TextBlob(str(text)).sentiment.polarity

    if polarity > 0.05:
        return "positive"
    if polarity < -0.05:
        return "negative"
    return "neutral"

df["textblob_sentiment"] = df["review_text_original"].apply(textblob_sentiment)
display(df["textblob_sentiment"].value_counts(normalize=True).rename("proportion"))

## 6.2 VADER

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

vader_analyzer = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    compound = vader_analyzer.polarity_scores(str(text))["compound"]

    if compound >= 0.05:
        return "positive"
    if compound <= -0.05:
        return "negative"
    return "neutral"

df["vader_sentiment"] = df["review_text_original"].apply(vader_sentiment)
display(df["vader_sentiment"].value_counts(normalize=True).rename("proportion"))

## 6.3 Transformer-Based Sentiment Analysis

A three-class transformer is used so its output can be compared directly with TextBlob and VADER. Adjust the batch size if memory is limited.

In [ ]:
from transformers import pipeline
from tqdm.auto import tqdm

transformer_sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True,
    max_length=512,
)

def normalize_transformer_label(label):
    label = str(label).strip().lower()
    if "positive" in label:
        return "positive"
    if "negative" in label:
        return "negative"
    return "neutral"

TRANSFORMER_BATCH_SIZE = 32
texts = df["review_text_original"].fillna("").tolist()
transformer_predictions = []

for start in tqdm(range(0, len(texts), TRANSFORMER_BATCH_SIZE)):
    batch = texts[start:start + TRANSFORMER_BATCH_SIZE]
    batch_predictions = transformer_sentiment_pipeline(
        batch,
        batch_size=TRANSFORMER_BATCH_SIZE,
    )
    transformer_predictions.extend(batch_predictions)

df["transformer_sentiment"] = [
    normalize_transformer_label(prediction["label"])
    for prediction in transformer_predictions
]

df["transformer_confidence"] = [
    prediction["score"]
    for prediction in transformer_predictions
]

display(df["transformer_sentiment"].value_counts(normalize=True).rename("proportion"))

## 6.4 Sentiment Distribution Comparison

In [ ]:
sentiment_distribution = pd.concat(
    {
        "TextBlob": df["textblob_sentiment"].value_counts(normalize=True),
        "VADER": df["vader_sentiment"].value_counts(normalize=True),
        "Transformer": df["transformer_sentiment"].value_counts(normalize=True),
    },
    axis=1,
).fillna(0)

sentiment_distribution = sentiment_distribution.reindex(
    ["positive", "neutral", "negative"]
)

display(sentiment_distribution)

sentiment_distribution.plot(kind="bar", figsize=(10, 6))
plt.title("Comparison of Sentiment Classification Distributions")
plt.xlabel("Sentiment")
plt.ylabel("Proportion of Reviews")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6.5 Human-Coded Sentiment Validation

Export a reproducible sample, have group members assign `human_sentiment`, then reload the completed file and compare model performance.

In [ ]:
HUMAN_SAMPLE_SIZE = 100

human_validation_sample = df[
    [
        "review_text_original",
        "textblob_sentiment",
        "vader_sentiment",
        "transformer_sentiment",
    ]
].sample(
    n=min(HUMAN_SAMPLE_SIZE, len(df)),
    random_state=RANDOM_STATE,
).copy()

human_validation_sample["human_sentiment"] = ""
human_validation_file = OUTPUT_PATH / "sentiment_human_validation_sample.csv"
human_validation_sample.to_csv(human_validation_file, index=False)

print("Human validation sample saved to:", human_validation_file)
display(human_validation_sample.head())

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# After the human-coded file is completed, set this path and run the evaluation.
COMPLETED_HUMAN_VALIDATION_FILE = OUTPUT_PATH / "sentiment_human_validation_completed.csv"

if COMPLETED_HUMAN_VALIDATION_FILE.exists():
    human_df = pd.read_csv(COMPLETED_HUMAN_VALIDATION_FILE)
    human_df["human_sentiment"] = human_df["human_sentiment"].str.strip().str.lower()
    human_df = human_df[
        human_df["human_sentiment"].isin(["positive", "neutral", "negative"])
    ].copy()

    for model_column in [
        "textblob_sentiment",
        "vader_sentiment",
        "transformer_sentiment",
    ]:
        print(f"\n{model_column}")
        print("Accuracy:", accuracy_score(
            human_df["human_sentiment"],
            human_df[model_column],
        ))
        print(classification_report(
            human_df["human_sentiment"],
            human_df[model_column],
            zero_division=0,
        ))
else:
    print(
        "Complete the human-coded validation sample, save it as "
        "'sentiment_human_validation_completed.csv', and rerun this cell."
    )

## 6.6 Sentiment by Best Topic Model

The example below uses BERTopic. Replace `best_sentiment_column` after human validation confirms the most accurate sentiment method.

In [ ]:
best_sentiment_column = "transformer_sentiment"

bertopic_results = pd.DataFrame({
    "review_text_original": bertopic_documents,
    "topic_id": bertopic_topics,
})

# Match sentiment results back to original review text.
sentiment_lookup = (
    df[["review_text_original", best_sentiment_column]]
    .drop_duplicates("review_text_original")
)

bertopic_results = bertopic_results.merge(
    sentiment_lookup,
    on="review_text_original",
    how="left",
)

topic_sentiment_summary = (
    bertopic_results[bertopic_results["topic_id"] != -1]
    .groupby(["topic_id", best_sentiment_column])
    .size()
    .unstack(fill_value=0)
)

topic_sentiment_percent = topic_sentiment_summary.div(
    topic_sentiment_summary.sum(axis=1),
    axis=0
)

display(topic_sentiment_percent.head(15))

# 7. Text Classification

The goal is to predict recommendation status from review text. Logistic Regression is used as the baseline and compared with Naive Bayes, Linear SVM, and Random Forest. Class weighting is used where supported.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

classification_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

classification_results = []
classification_predictions = {}

for model_name, model in classification_models.items():
    model.fit(X_train_tfidf, y_train)
    predictions = model.predict(X_test_tfidf)
    classification_predictions[model_name] = predictions

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test,
        predictions,
        average="weighted",
        zero_division=0,
    )

    classification_results.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision_weighted": precision,
        "recall_weighted": recall,
        "f1_weighted": f1,
    })

classification_results_df = pd.DataFrame(classification_results)
classification_results_df = classification_results_df.sort_values(
    "f1_weighted",
    ascending=False,
)

display(classification_results_df)

## 7.1 Classification Reports

In [ ]:
for model_name, predictions in classification_predictions.items():
    print("=" * 80)
    print(model_name)
    print(classification_report(y_test, predictions, zero_division=0))

## 7.2 Confusion Matrices

In [ ]:
for model_name, predictions in classification_predictions.items():
    matrix = confusion_matrix(y_test, predictions)

    display_object = ConfusionMatrixDisplay(
        confusion_matrix=matrix,
        display_labels=["Not Recommended", "Recommended"],
    )

    display_object.plot()
    plt.title(f"Confusion Matrix: {model_name}")
    plt.tight_layout()
    plt.show()

## 7.3 Optional Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

logistic_parameter_grid = {
    "C": [0.1, 0.5, 1.0, 2.0, 5.0],
    "solver": ["liblinear", "saga"],
}

logistic_grid_search = GridSearchCV(
    estimator=LogisticRegression(
        max_iter=2500,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    param_grid=logistic_parameter_grid,
    scoring="f1_weighted",
    cv=5,
    n_jobs=-1,
    verbose=1,
)

logistic_grid_search.fit(X_train_tfidf, y_train)

print("Best parameters:", logistic_grid_search.best_params_)
print("Best cross-validation score:", logistic_grid_search.best_score_)

tuned_logistic_predictions = logistic_grid_search.predict(X_test_tfidf)
print(classification_report(
    y_test,
    tuned_logistic_predictions,
    zero_division=0,
))

## 7.4 Optional XGBoost

Run this section only when XGBoost is installed and computational resources permit.

In [ ]:
# from xgboost import XGBClassifier
#
# xgb_model = XGBClassifier(
#     n_estimators=300,
#     max_depth=6,
#     learning_rate=0.05,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     objective="binary:logistic",
#     eval_metric="logloss",
#     random_state=RANDOM_STATE,
#     n_jobs=-1,
# )
#
# xgb_model.fit(X_train_tfidf, y_train)
# xgb_predictions = xgb_model.predict(X_test_tfidf)
# print(classification_report(y_test, xgb_predictions, zero_division=0))

## 7.5 Most Influential Classification Terms

Logistic Regression coefficients help identify words and phrases associated with recommended and not-recommended reviews.

In [ ]:
baseline_logistic = classification_models["Logistic Regression"]
feature_names = np.array(tfidf_classifier.get_feature_names_out())
coefficients = baseline_logistic.coef_[0]

top_positive_indices = coefficients.argsort()[-25:][::-1]
top_negative_indices = coefficients.argsort()[:25]

positive_terms = pd.DataFrame({
    "term": feature_names[top_positive_indices],
    "coefficient": coefficients[top_positive_indices],
})

negative_terms = pd.DataFrame({
    "term": feature_names[top_negative_indices],
    "coefficient": coefficients[top_negative_indices],
})

print("Terms associated with recommendation:")
display(positive_terms)

print("Terms associated with non-recommendation:")
display(negative_terms)

# 8. Model Comparison and Integrated Results

In [ ]:
print("Classification comparison")
display(classification_results_df)

print("\nTraditional topic-model comparison")
display(traditional_topic_comparison.sort_values("coherence_cv", ascending=False))

print("\nSentiment distribution comparison")
display(sentiment_distribution)

In [ ]:
# Save primary result tables.
classification_results_df.to_csv(
    OUTPUT_PATH / "classification_model_comparison.csv",
    index=False,
)

traditional_topic_comparison.to_csv(
    OUTPUT_PATH / "topic_model_comparison.csv",
    index=False,
)

sentiment_distribution.to_csv(
    OUTPUT_PATH / "sentiment_distribution_comparison.csv"
)

topic_sentiment_percent.to_csv(
    OUTPUT_PATH / "sentiment_by_bertopic_topic.csv"
)

print("Result tables saved in:", OUTPUT_PATH.resolve())

# 9. Conclusions, Recommendations, Limitations, and Future Research

Complete this section after all models have been evaluated.

## Key Findings

1. **Most common passenger topics:**  
   Add the final topic labels and descriptions.

2. **Overall sentiment:**  
   Add the final positive, neutral, and negative distribution from the selected sentiment model.

3. **Best topic model:**  
   Identify the selected model using coherence, interpretability, and human review.

4. **Best sentiment model:**  
   Identify the model with the strongest human-coded validation performance.

5. **Best recommendation classifier:**  
   Identify the model with the strongest test-set F1 score and discuss its confusion matrix.

## Business Recommendations

- Prioritize service areas associated with strongly negative topics.
- Improve disruption and delay communication.
- Strengthen consistency in cabin crew and ground-service interactions.
- Address recurring concerns involving seating, boarding, baggage, and value for money.
- Track topic-level sentiment over time to measure whether corrective actions improve customer perceptions.

## Limitations

- Reviews are self-selected and may overrepresent extreme experiences.
- Skytrax reviewers may not represent all airline passengers.
- Older reviews may describe services that have since changed.
- Automated sentiment methods may misinterpret sarcasm, mixed opinions, and aviation-specific context.
- Topic labels require human interpretation.
- English-language preprocessing may exclude or distort multilingual feedback.

## Future Research

- Conduct airline-level and regional comparisons.
- Apply aspect-based sentiment analysis.
- Evaluate trends across time periods.
- Compare verified and unverified reviews if that variable is available.
- Use large language models for topic labeling and qualitative summarization.

# 10. Reproducibility Checklist

Before submission:

- Confirm that the notebook runs from top to bottom.
- Record the exact number of rows before and after cleaning.
- Replace placeholder interpretations with group-authored explanations.
- Confirm all visualizations include titles, labels, and written interpretations.
- Complete human coding for sentiment and topic validation.
- Export the notebook as both `.ipynb` and HTML or PDF.
- Submit the original and cleaned datasets with the final project files.